Note we utilize duckdb view chains to modify the large dataset

If you are unfamiliar with duckdb please refer to the documentation: https://duckdb.org/docs/stable/clients/python/overview

DOWNLOAD: This file must be moved from Snellius to its corresponding local folder. Ignore this tag for local processing purposes.

# Data Loading

In [ ]:
snellius = True  # Set to True if running on Snellius, False for local development
partition = "operational_features_process" if snellius else "subset_week"

In [ ]:
create_dataset = True  # Set to True to create the dataset, False to skip for faster execution
check_stats = False  # cSet to True to display intermediate stats, False to skip for faster execution
create_intermediate_datasets = False
check_rowc = True

In [43]:
rstats = (snellius is True) and (check_stats is True) #run stats
cdata_new = (snellius is True) and (create_dataset is True) # create data
cdata_old = (cdata_new is True) and (create_intermediate_datasets is True) 

## Set root folder path 

In [44]:
# pip install any missing packages before running the notebook
import duckdb
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import networkx as nx

In [45]:
import os
from pathlib import Path

if snellius is False:
    # Change later to snellius $HOME or whatever folder you want to work in
    target_root = Path(r"C:\Users\jialo\Desktop\MSc-DS-Thesis\MSc-Thesis-Repo\MSc-Thesis\Bao")

    # Check if it exists before moving
    if target_root.exists():
        os.chdir(target_root)
        print(f"✅ Success! Moved to: {Path.cwd()}")
    else:
        print(f"❌ Error: The folder '{target_root}' does not exist.")


✅ Success! Moved to: C:\Users\jialo\Desktop\MSc-DS-Thesis\MSc-Thesis-Repo\MSc-Thesis\Bao


## Snellius Setup
We can directly read and write files to our personal $HOME node on snellius (instead of the tmpdir and copying back and forth)

To run this notebook, utilize the same project structure from /Bao onwards

First transfer files to Snellius (vscode) terminal:
scp -r ./MSc-Thesis/Bao/X username@snellius.surf.nl:~/Project_folder/X
where X = {all_code, downloads, media} folders

(or gitbash):
rsync -avP ./MSc-Thesis/Bao/{all_code,downloads,media} name@snellius.surf.nl:~/NS_Thesis/

On Snellius:
setup environment.yml
run sbatch InstallEnv.job
run the corresponding .job

In [46]:
if snellius is True:
    from pathlib import Path
    # Get the home directory path object
    home_dir = Path.home() 

    # Build a path to your code
    target_root = home_dir / "NS_Thesis"

    print(target_root)

    # Test writing a file to confirm the path works
    # # 1. Define the path (Home Directory)
    # # On Snellius, this will resolve to /home/jbao
    # file_path = Path.home() / "test_file.txt"

    # # 2. Write a small test message
    # try:
    #     with open(file_path, "w") as f:
    #         f.write("This is a test file to verify the home directory path.\n")
        
    #     print(f"✅ Success! File saved to: {file_path}")
    #     print("⚠️  Reminder: Do not save large datasets here (16GB Quota).")
    # except Exception as e:
    #     print(f"❌ Error: Could not write to file. {e}")

## Set paths

In [47]:
project_root = target_root

root_folder =  project_root / "downloads/data/raw/"
filtered_output_root_folder = project_root / "downloads/data/filtered/"

services_path = root_folder / "NS-services/services_merged_all_ns_only.parquet"
disruptions_path = root_folder / "NS-disruptions/disruptions_merged_all.parquet"
stations_path = root_folder / "NS-stations/stations-2023-09-nl.csv" #duckdb seem to handle 'NA' station code properly, so load the original file
station_distances_path = root_folder / "NS-tariff-distances/tariff-distances-2022-01.csv" #probably not used in eda, might only be useful for graph edges feature
stations_connections_path = root_folder / "railway_map/connection_edges.parquet" 

weather_path = root_folder / "weather/weather_merged_all.parquet"
holiday_path = root_folder / "holidays/dutch_holidays_2019_2025.parquet"

media_folder = project_root / "media"
# if not media_folder.exists():
#     media_folder.mkdir()

# 1. Setup paths to iterate over

paths = {
    "Services": services_path,
    "Disruptions": disruptions_path,
    "Stations": stations_path,
    "Station Distances": station_distances_path,
    "Weather": weather_path,
    "Holidays": holiday_path
}


# 2. Paths of filtered datasets (after EDA and cleaning)
aggregated_hourly_path = filtered_output_root_folder / "final_aggregated/services_hourly.parquet"

GRAPH_OUTPUT_DIR = filtered_output_root_folder / "graph_features_output"
os.makedirs(GRAPH_OUTPUT_DIR, exist_ok=True)

graph_features_path = filtered_output_root_folder / "final_aggregated/services_hourly_graph_features.parquet"

In [48]:
## Connect to database
db_name = 'main_thesis_data.duckdb'

# Check before
print(f"Does file exist? {os.path.exists(db_name)}")

con = duckdb.connect(database='main_thesis_data.duckdb') 

# Optional: Explicitly cap RAM so DuckDB knows WHEN to start spilling
# (e.g., set to 70-80% of your actual machine RAM)
if snellius is False:
    con.execute("SET memory_limit='5GB'")   

Does file exist? True


## SAVING CODE

duckdb tables to parquet

In [49]:
# aggregated_hourly = filtered_output_root_folder / "final_aggregated/services_hourly.parquet"
# con.execute(f"COPY services_hourly_agg TO '{aggregated_hourly}' (FORMAT PARQUET)")
# print("Done! Saved as ", aggregated_hourly)

## Helper functions

In [50]:
def check_arrival_delay_distribution(dataset_name):
    print(f"Checking arrival_delay_min distribution for {dataset_name}...")
    ad_distribution_df = con.execute(f"""
        SELECT 
            -- Create the bin floor (0, 100, 200...)
            FLOOR(arrival_delay_min / 100) * 100 AS bin_start,
            
            -- Create a readable label
            CAST(FLOOR(arrival_delay_min / 100) * 100 AS INTEGER) || ' to ' || 
            CAST(FLOOR(arrival_delay_min / 100) * 100 + 100 AS INTEGER) || ' min' AS bin_label,
            
            -- Count services in this bin
            COUNT(*) AS count,
            
            -- Optional: Percentage
            ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM {dataset_name} WHERE arrival_delay_min IS NOT NULL), 4) AS pct
            
        FROM {dataset_name}
        WHERE arrival_delay_min IS NOT NULL
        GROUP BY 1, 2
        ORDER BY 1
    """).df()

    return display(ad_distribution_df)

def count_row_diff(data_before, data_after):
	print(f"Checking rows for {data_before} and {data_after}...")

	rows_df = con.execute(f"""
		SELECT 
			(SELECT COUNT(*) FROM {data_before}) AS original_rows,
			(SELECT COUNT(*) FROM {data_after}) AS remaining_rows,
			(SELECT COUNT(*) FROM {data_before}) - (SELECT COUNT(service_id) FROM {data_after}) AS dropped_rows
	""").df()
	return display(rows_df)

## Duckdb prep

In [51]:
# 2. Initialize Disk-Based Database
# This creates a file 'main_thesis_data.duckdb' in your current folder.
# Intermediate calculations (joins/group bys) will spill here instead of crashing RAM.
# con = duckdb.connect(database='main_thesis_data.duckdb') 

# 3. Register Views (Zero-Copy), Run once
# We use VIEWs so we don't duplicate the Parquet data into the .duckdb file.
# DuckDB reads directly from the parquet files on demand.
con.execute(f"CREATE OR REPLACE VIEW raw_services AS SELECT * FROM read_parquet('{services_path}')")
con.execute(f"CREATE OR REPLACE VIEW raw_disruptions AS SELECT * FROM read_parquet('{disruptions_path}')")
con.execute(f"CREATE OR REPLACE VIEW raw_weather AS SELECT * FROM read_parquet('{weather_path}')")
con.execute(f"CREATE OR REPLACE VIEW stations AS SELECT * FROM read_csv_auto('{stations_path}')")
con.execute(f"CREATE OR REPLACE VIEW raw_holidays AS SELECT * FROM read_parquet('{holiday_path}')")
con.execute(f"CREATE OR REPLACE VIEW raw_station_distances AS SELECT * FROM read_csv_auto('{station_distances_path}')")
con.execute(f"CREATE OR REPLACE VIEW raw_stations_connections AS SELECT * FROM read_parquet('{stations_connections_path}')") # Prob not needed as stations from RdT is pulled from NS API already, but just in case

# Aggregated dataset for graph edges features
con.execute(f"CREATE OR REPLACE VIEW aggregated_hourly AS SELECT * FROM read_parquet('{aggregated_hourly_path}')")
con.execute(f"CREATE OR REPLACE VIEW aggregated_hourly_graph_features AS SELECT * FROM read_parquet('{graph_features_path}')")




# 4. Create CLEAN Views (Logic remains the same, but executed safely)
con.execute("""
    CREATE OR REPLACE VIEW services AS 
    SELECT 
        "Service:RDT-ID" AS service_id,
        "Service:Date" AS service_date,
        "Service:Type" AS train_type,
            
        "Service:Completely cancelled" AS is_completely_cancelled, 
        "Service:Partly cancelled" AS is_partly_cancelled,

        "Stop:Station code" AS station_code,
        "Stop:Station name" AS station_name,
        
        -- Apply Timezone conversion HERE in the SELECT statement
        ("Stop:Arrival time" AT TIME ZONE 'Europe/Amsterdam')::TIMESTAMP AS arrival_time,
        
        "Stop:Arrival delay" AS arrival_delay_min,
        "Stop:Arrival cancelled" AS is_arrival_cancelled,
        
        -- Apply Timezone conversion HERE too
        ("Stop:Departure time" AT TIME ZONE 'Europe/Amsterdam')::TIMESTAMP AS departure_time,
        
        "Stop:Departure delay" AS departure_delay_min,
        "Stop:Departure cancelled" AS is_departure_cancelled,
        
        "Stop:Platform change" AS has_platform_change,
            
    FROM raw_services
""")

print("✅ Table 'services' created with clean names and local timestamps.")

con.execute("""
    CREATE OR REPLACE VIEW disruptions AS 
    SELECT 
        *,
        CASE 
            WHEN rdt_station_codes LIKE '[%]' 
            THEN string_split(trim(rdt_station_codes, '[]'), ', ') 
            ELSE string_split(rdt_station_codes, ',') 
        END AS station_array
    FROM raw_disruptions
""")

print("Disk-based database initialized. Queries will now spill to 'main_thesis_data.duckdb' if needed.")

✅ Table 'services' created with clean names and local timestamps.
Disk-based database initialized. Queries will now spill to 'main_thesis_data.duckdb' if needed.


In [52]:
con.execute("""DESCRIBE aggregated_hourly_graph_features""").df()

,column_name,column_type,null,key,default,extra
0,source,VARCHAR,YES,None,None,None
1,target,VARCHAR,YES,None,None,None
2,service_date,DATE,YES,None,None,None
3,arrival_hour,BIGINT,YES,None,None,None
4,earliest_departure_hour,BIGINT,YES,None,None,None
5,distance,INTEGER,YES,None,None,None
6,total_departure_delay_minutes,DOUBLE,YES,None,None,None
7,avg_departure_delay_minutes,DOUBLE,YES,None,None,None
8,count_delayed_departure_services,BIGINT,YES,None,None,None
9,total_delay_minutes,DOUBLE,YES,None,None,None


## Total services in dataset

Total services ROWCOUNT = 83_816_585

In [53]:
if rstats:
	display(con.execute("SELECT count(service_id) FROM services").df())

# Processing
from duckdb EDA

## Preprocess: NA values Cleaning and Handling
### NA in Services Arrival and Departures Columns
In short the meaning of NA values in services arrival and departure columns indicate either unscheduled services, a start station or terminating stations. Which is deduced from the rdt documentation "Each row in these files represent a stop at a station. Each service at least departs from and arrives at a station (i.e. two rows). For each stop, you can find the name of the station, the arrival and departure time, delays and cancellations. The exact meaning of each column is explained below." https://www.rijdendetreinen.nl/en/open-data/train-archive

In [54]:
if rstats:
	# Check if there are cases where arrival_delay_min is present but arrival_time is NULL (which would mean that a train was delayed but not scheduled to arrive)
	display(con.execute("""
		SELECT 
			service_id,
			station_name,
			arrival_time,
			arrival_delay_min,
			is_arrival_cancelled
		FROM services
		WHERE 
			arrival_delay_min IS NOT NULL 
			AND arrival_time IS NULL
		LIMIT 20
	""").df())

In [55]:
if rstats:
	# Query to find "Terminus" stops (Arrival exists, Departure missing)
	df_terminus = con.execute("""
		SELECT 
			service_id,
			station_name,
			arrival_delay_min,
			arrival_time,
			is_arrival_cancelled
		FROM services
		WHERE 
			-- Condition 1: Departure info is missing (OR logic as requested)
			(arrival_time IS NULL OR arrival_delay_min IS NULL OR is_arrival_cancelled IS NULL)
			
			AND 
			
			-- Condition 2: Arrival info is present (OR logic as requested)
			(departure_time IS NOT NULL OR departure_delay_min IS NOT NULL OR is_departure_cancelled IS NOT NULL)
		
		-- Optional: Limit to see a sample first
		LIMIT 20
	""").df()

	display(df_terminus)

#### p.1) Handling NA values in the services dataset
- Remove unscheduled trains (NA at both arrival and departure columns)
- Fill start stations (NA at arrival columns, it never 'arrives' so columns:arrival_time = departure_time, arrival_delay_min = 0, is_arrival_cancelled = FALSE)
- Removing end stations

We note that:

ALL NA values in both arrival and departure columns indicate a unscheduled service

ALL NA values in arrival_time and arrival_delay_min indicate the start of a service.

ALL NA values in departure_time and arrival_delay_min indicate the end of a service. 

A starting station could never have arrival delay as it is already there (IS NA values in arrival columns)

A end station could never have departure delay as will never depart there (IS NA values in departure columns)

By handling the NA values like this we add some biases to our dataset, but it logically follows from the schedule.

##### A) Removing unscheduled services (services with ONLY NA values in both departure and arrival time & delay) 
REMOVED: 5652 services

ROWCOUNT services_scheduled = 83_810_933

In [56]:
con.execute("""
    CREATE OR REPLACE VIEW services_scheduled AS 
    SELECT * FROM services
    WHERE NOT (
        arrival_time IS NULL 
        AND departure_time IS NULL 
        AND arrival_delay_min IS NULL
        AND departure_delay_min IS NULL
    )
""")

print("✅ View 'services_scheduled' created (Unscheduled rows hidden).")

✅ View 'services_scheduled' created (Unscheduled rows hidden).


In [57]:
if check_rowc:
    count_row_diff("services", "services_scheduled")

In [58]:
if rstats:
    display(con.execute("SELECT count(service_id) FROM services_scheduled").df())

##### B) We impute the values for the starting station (NA values for ALL arrival columns) (8_121_562 services) and REMOVE the end stations (NA values for ALL departure columns) (8_119_198 services)

2364 train services seem to be missing end stations, this is most be explained by the column "Service:Train number" due to merging (or splitting) of trains, as the following is mentioned in the dataset documentation:
"A single service may sometimes have multiple train numbers. For example, when a train is split in two parts, or when a train changes a train number on a major station halfway." https://www.rijdendetreinen.nl/en/open-data/train-archive
We leave these in as we are analysing stop trajectories in our train railway network, not specific service train trajectories

In [59]:
con.execute("""
    CREATE OR REPLACE VIEW services_filled AS 
    SELECT 
        * REPLACE (
            -- =========================================================
            -- 1. STRICT Fix for Start Stations (Fill Arrival)
            -- Only runs if ALL 3 arrival columns are NULL
            -- =========================================================
            CASE 
                WHEN arrival_time IS NULL 
                     AND arrival_delay_min IS NULL 
                     AND is_arrival_cancelled IS NULL
                THEN departure_time             -- Action: Fill with Departure
                ELSE arrival_time               -- Action: Keep original (even if NULL)
            END AS arrival_time,

            CASE 
                WHEN arrival_time IS NULL 
                     AND arrival_delay_min IS NULL 
                     AND is_arrival_cancelled IS NULL
                THEN 0                          -- Action: Assume 0 delay
                ELSE arrival_delay_min          -- Action: Keep original
            END AS arrival_delay_min,

            CASE 
                WHEN arrival_time IS NULL 
                     AND arrival_delay_min IS NULL 
                     AND is_arrival_cancelled IS NULL
                THEN FALSE                      -- Action: Assume not cancelled
                ELSE is_arrival_cancelled       -- Action: Keep original
            END AS is_arrival_cancelled
        )
    FROM services_scheduled 
    -- =========================================================
    -- 2. REMOVE rows where ALL departure columns are NULL
    -- =========================================================
    WHERE NOT (
        departure_time IS NULL 
        AND departure_delay_min IS NULL 
        AND is_departure_cancelled IS NULL
    )
""")

print("✅ View 'services_filled' created. Strict imputation for arrivals applied, and missing departures dropped.")

✅ View 'services_filled' created. Strict imputation for arrivals applied, and missing departures dropped.


Should be 8_119_198 rows dropped

In [60]:
if check_rowc:
    count_row_diff("services_scheduled", "services_filled")

#### p.2) Arrival Delay of Cancellations

In short we remap the 7_813_345 cancelled services as delayed services

We set an arrival delay penalty of 30 min to partly cancelled services and 60 min to completely cancelled services

Cancellations as % of all services

In [61]:
if rstats:
	con.execute("""
		SELECT 
			-- Total
			count(*) as total_services,
			
			-- Counts
			count(*) FILTER (WHERE is_partly_cancelled OR is_completely_cancelled) as total_all_cancelled_services,
			count(*) FILTER (WHERE is_partly_cancelled AND NOT is_completely_cancelled) as count_partly_cancelled,
			count(*) FILTER (WHERE is_completely_cancelled) as count_completely_cancelled,
			count(*) FILTER (WHERE is_partly_cancelled AND is_completely_cancelled) as count_both_cancelled,
				
				
			-- Percentages (Logic repeated)
			round(
				total_all_cancelled_services / total_services * 100.0, 
			2) as pct_all_cancelled,

			round(
				count_partly_cancelled / total_services * 100.0, 
			2) as pct_partly_cancelled,

			round(
				count_completely_cancelled / total_services * 100.0, 
			2) as pct_completely_cancelled,
				
			round(
				count_both_cancelled / total_services * 100.0,
			2) as pct_both_cancelled
			
		FROM services_filled
	""").df()

Cancellations as % of all CANCELLED services

In [62]:
if rstats:
	con.execute("""
		SELECT 
			-- Total
			-- count(*) as total_services,
			
			-- Counts
			count(*) FILTER (WHERE is_partly_cancelled OR is_completely_cancelled) as total_all_cancelled_services,
			count(*) FILTER (WHERE is_partly_cancelled AND NOT is_completely_cancelled) as count_partly_cancelled,
			count(*) FILTER (WHERE is_completely_cancelled) as count_completely_cancelled,
			count(*) FILTER (WHERE is_partly_cancelled AND is_completely_cancelled) as count_both_cancelled,
				
			-- Percentages (Logic repeated)
			round(
				total_all_cancelled_services / total_all_cancelled_services * 100.0, 
			2) as pct_all_cancelled,

			round(
				count_partly_cancelled / total_all_cancelled_services * 100.0, 
			2) as pct_partly_cancelled,

			round(
				count_completely_cancelled / total_all_cancelled_services * 100.0, 
			2) as pct_completely_cancelled,
			
			round(
				count_both_cancelled / total_all_cancelled_services * 100.0,
			2) as pct_both_cancelled
		
			
		FROM services_filled
	""").df()

9.3% (7_813_345 million services) of all services (83_810_933) are affected by cancellations

out of all cancelled services

76.1% is partly cancelled

23.9% is completely cancelled

In [63]:
print(7_813_345 / 83_810_933 * 100)

9.322584441340128


In [64]:
if rstats:
	con.execute("""
		SELECT 
			-- Total Cancelled Rows
			count(*) as total_cancelled,
			
			-- 1. Delay >= 60 mins
			count(*) FILTER (WHERE arrival_delay_min >= 60) as count_gt_60,
			round(count(*) FILTER (WHERE arrival_delay_min >= 60) * 100.0 / count(*), 2) as pct_gt_60,

			-- 2. Delay >= 30 < 60 mins
			count(*) FILTER (WHERE arrival_delay_min >= 30 AND arrival_delay_min < 60) as count_gt_30,
			round(count(*) FILTER (WHERE arrival_delay_min >= 30 AND arrival_delay_min < 60) * 100.0 / count(*), 2) as pct_gt_30,
			
			-- 2. Delay > 0 < 30 mins
			count(*) FILTER (WHERE arrival_delay_min > 0 AND arrival_delay_min < 30) as count_gt_1,
			round(count(*) FILTER (WHERE arrival_delay_min > 0 AND arrival_delay_min < 30) * 100.0 / count(*), 2) as pct_gt_1,

			-- 3. Delay <= 0
			count(*) FILTER (WHERE arrival_delay_min <= 0) as count_eq_0,
			round(count(*) FILTER (WHERE arrival_delay_min <= 0) * 100.0 / count(*), 2) as pct_eq_0,
				
		FROM services_filled
		WHERE 
			-- Focus only on cancelled services
			(is_partly_cancelled = TRUE OR is_completely_cancelled = TRUE)
	""").df()

##### p.2A) Set Arrival Delay Penalty for both completely and partly cancelled services (changes the delayed classification distribution)
The exact number doesnt matter, is significantly delay target label based on number of services on trajectory.
We essentially count these cancelled and partly cancelled services as delayed services, taking into account the passenger's perspective instead of completely dropping these relevant services

In [65]:
con.execute("""
    CREATE OR REPLACE VIEW services_penalty AS 
    SELECT * REPLACE (
        CASE
            WHEN is_completely_cancelled = TRUE THEN 60
            WHEN is_partly_cancelled = TRUE THEN 30
            ELSE arrival_delay_min
        END AS arrival_delay_min
    )
    FROM services_filled
""")

# Verify the "overwrite" worked
if rstats:
	con.execute("SELECT is_completely_cancelled, arrival_delay_min FROM services_penalty WHERE is_completely_cancelled = TRUE LIMIT 5").df()

In [66]:
if check_rowc:
    count_row_diff("services_filled", "services_penalty")

In [67]:
if rstats:
	display(con.execute("SELECT * FROM services_penalty LIMIT 10").df())

could also exclude all services with cancellations from the services dataset with the commented code below, but this ignores passengers perspective

In [68]:
# con.execute("""
#     CREATE OR REPLACE VIEW services_penalty AS 
#     SELECT * FROM services_filled
#     WHERE 
#         -- Exclude cancelled arrivals
#         is_arrival_cancelled IS NOT TRUE
        
#         -- Exclude cancelled departures
#         AND is_departure_cancelled IS NOT TRUE
# """)

# # Quick verification: Check how many rows remain
# print(con.execute("SELECT count(*) FROM services_penalty").fetchone()[0])

In [69]:
if snellius is True:
    # Checking arrival_delay_min distribution after penalty application
    check_arrival_delay_distribution("services_penalty")

## Transformation 1: adding 'to stations' columns to services dataset

Each service has atleast two rows: the departure (current Stop:Station, i.e. from station) and arrival (to station inferred from sorting the service line and time). 

source = departure station

target = arrival station

In [70]:
# ==========================================
# STEP 2: Create Edges (Window Functions)
# ==========================================
# We materialize this as a TABLE to compute the expensive window functions once.
con.execute("""
    CREATE OR REPLACE VIEW services_with_edges AS 
    SELECT 
        *,
        -- From station is just the current row's station code
        station_code AS source,
            
        -- 1. Where are we going next? Get Destination (Next Row)
        LEAD(station_code) OVER (
            PARTITION BY service_id 
            ORDER BY COALESCE(departure_time, arrival_time) ASC
        ) AS target

    FROM services_penalty
""")
print("✅ Layer 2: Added Edge columns.")

✅ Layer 2: Added Edge columns.


Edges = 2931 (takes ~1min locally)

In [71]:
if rstats:
    display(con.execute("""
        SELECT count(*) AS unique_edges
        FROM (
            SELECT DISTINCT source, target 
            FROM services_with_edges
            WHERE target IS NOT NULL
        )
    """).df())

## Transformation 2: Stations
We keep the stations dataset as ground truth for which stations exist (in the netherlands) and drop services with other stations

### A) Add station distances feature

In [72]:
# ==========================================
# STEP 1: Flatten distance matrix dataset
# ==========================================
con.execute("""
    CREATE OR REPLACE VIEW station_distances_long AS 
    SELECT 
        Station AS source,          -- Rename 'Station' to 'source' for clarity
        target_station AS target, 
        CAST(distance AS INTEGER) AS distance
    FROM (
        UNPIVOT raw_station_distances
        ON COLUMNS(* EXCLUDE (Station))
        INTO
            NAME target_station
            VALUE distance
    )
    -- Filter 1: Explicitly remove the known garbage values
    WHERE distance NOT IN ('?', 'XXX')
    
    -- Filter 2: (Optional but Safer) Ensure only valid numbers remain
    -- AND TRY_CAST(distance AS INTEGER) IS NOT NULL
""")

if rstats:
	# Let's verify the columns first to be safe:
	print("Unpivoted columns:", con.execute("DESCRIBE station_distances_long").df()['column_name'].tolist())

Following 2 stations from the distance connection matrix were not in stations

LEER
WR

In [73]:
if rstats:
	con.execute("""
		-- 1. Get ALL stations from the matrix
		WITH matrix_stations AS (
			SELECT TRIM(source) AS station FROM station_distances_long
			UNION 
			SELECT TRIM(target) AS station FROM station_distances_long
		)
		
		-- 2. "Subtract" the official list
		SELECT station AS not_in_stations FROM matrix_stations
		
		EXCEPT 
		
		SELECT TRIM(code) FROM stations
	""").df()

In [74]:
if rstats:
	con.execute("""
		SELECT TRIM(code) AS not_in_distances
		FROM stations
		
		EXCEPT 
		
		-- Subtract all stations found in the matrix
		(
			SELECT TRIM(source) FROM station_distances_long
			UNION 
			SELECT TRIM(target) FROM station_distances_long
		)
	""").df()

In [75]:
# ==========================================
# STEP 2: Join Distances
# ==========================================
con.execute("""
    CREATE OR REPLACE VIEW services_edges_distances AS 
    SELECT 
        e.*, 
        d.distance AS distance
    FROM services_with_edges e
    LEFT JOIN station_distances_long d
        ON e.source = d.source 
        AND e.target = d.target
""")

print("✅ Layer 2A: Distances joined.")

✅ Layer 2A: Distances joined.


330 edges missing distances in services i.e. not in stations distances dataset ~(1 min)

TODO: remove missing distances in services in the end after monthly aggregation

In [76]:
if rstats:
	# Check for "Orphan Edges" (Edges that exist but have no distance)
	orphan_edges = con.execute("""
		SELECT DISTINCT source, target 
		FROM services_edges_distances
		WHERE 
			distance IS NULL    -- Capture where the join failed
	""").df()

	if not orphan_edges.empty:
		print(f"⚠️ Warning: {len(orphan_edges)} routes are missing distance data.")
		display(orphan_edges.head())
	else:
		print("✅ Success: All edges have a corresponding distance.")

### B) Services with valid stations (i.e. occuring in 2023-09 stations rdt dataset)

In [77]:
# ==========================================
# Filter Invalid Rows
# ==========================================
con.execute("""
    CREATE OR REPLACE VIEW services_valid AS 
    SELECT * FROM services_edges_distances
    WHERE 
        -- 1. Distance must be known (removes missing edges from the Matrix)
        distance IS NOT NULL
        
        -- 2. Source must be in the official station list
        AND (source IN (SELECT code FROM stations)
        
        -- 3. Target must be in the official station list
        AND target IN (SELECT code FROM stations))
""")

print("✅ Layer 2B: Filtered services to valid edges only.")

✅ Layer 2B: Filtered services to valid edges only.


Valid services ROWCOUNT = 
75_303_052 

In [78]:
if snellius is True:
    con.sql("SELECT count(*) FROM services_valid").show()

In [79]:
if rstats:
    check_arrival_delay_distribution("services_valid")

### Service delay bins (figures)

In [80]:
# con.execute("""
#     CREATE OR REPLACE VIEW delay_bins AS
#     SELECT 
#         -- Create the bins on the fly (or use your view)
#         CASE 
#             WHEN arrival_delay_min <= 0 THEN 'On Time (inf, 0]'
#             WHEN arrival_delay_min < 30 THEN 'Small Delay (0, 30)'
#             WHEN arrival_delay_min < 60 THEN 'Medium Delay [30, 60)'
#             ELSE 'Large Delay [60, inf)'
#         END AS delay_category,
        
#         -- Helper column for sorting the chart correctly
#         CASE 
#             WHEN arrival_delay_min <= 0 THEN 1
#             WHEN arrival_delay_min < 30 THEN 2
#             WHEN arrival_delay_min < 60 THEN 3
#             ELSE 4
#         END AS sort_order,

#         -- Count services in each bin
#         COUNT(*) as count
#     FROM services_valid
#     WHERE arrival_delay_min IS NOT NULL
#     GROUP BY 1, 2
#     ORDER BY sort_order
# """)

In [81]:
# binned_data_output_path = filtered_output_root_folder / "full_services_dataset/service_delay_bins.parquet"

In [82]:
# con.execute(f"COPY delay_bins TO '{binned_data_output_path}' (FORMAT PARQUET)")
# print("Done! Saved as ", binned_data_output_path)

In [83]:
# import matplotlib.pyplot as plt
# import pandas as pd

# # 1. Load Data
# binned_data_file = filtered_output_root_folder / "full_services_dataset/delay_bins.parquet"
# df_bins = pd.read_parquet(binned_data_file)

# # 2. Calculate Totals (Needed for %)
# total_services = df_bins['count'].sum()
# on_time_count = df_bins[df_bins['sort_order'] == 1]['count'].sum()
# delayed_count = df_bins[df_bins['sort_order'] > 1]['count'].sum()

# # 3. Setup the Plot Area
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# # --- Plot 1: Multi-Class Bar Chart ---
# bars = ax1.bar(
#     df_bins['delay_category'], 
#     df_bins['count'], 
#     color=['#2ca02c', '#ff7f0e', '#d62728', '#8c564b'], 
#     edgecolor='black', 
#     alpha=0.8
# )

# ax1.set_title('Distribution of Services Arrival Delays (Categorical)', fontsize=14)
# ax1.set_ylabel('Number of Services', fontsize=12)
# ax1.grid(axis='y', linestyle='--', alpha=0.3)

# # Add value labels AND percentages on top of bars
# for bar in bars:
#     height = bar.get_height()
#     percentage = (height / total_services) * 100  # Calculate %
    
#     # Format: "1,234 \n (12.5%)"
#     label_text = f'{int(height):,}\n({percentage:.1f}%)'
    
#     ax1.text(bar.get_x() + bar.get_width()/2., height,
#              label_text,
#              ha='center', va='bottom', fontsize=10, fontweight='bold')

# # --- Plot 2: Binary Pie Chart ---
# labels = [f'On Time\n({on_time_count:,})', f'Delayed\n({delayed_count:,})']
# sizes = [on_time_count, delayed_count]
# colors = ['#2ca02c', '#d62728'] 
# explode = (0, 0.1) 

# ax2.pie(sizes, explode=explode, labels=labels, colors=colors,
#         autopct='%1.1f%%', shadow=True, startangle=140, textprops={'fontsize': 11})
# ax2.set_title('Binary Classification: Services On Time vs Delayed', fontsize=14)

# # 4. Save and Show
# plt.tight_layout()
# # plt.savefig(f'{media_folder}/delay_classification_plots.png')
# # print("✅ Plots saved as 'delay_classification_plots.png'")

## Transformation 3: Monthly Aggregation of Trajectories 


We add binary classification column of is significantly delayed utilizing the median ratio delayed trajectory/total trajectories of total of the dataset

(Merel's and Jonathan paper utilized the COUNT of services on a trajectory for their classification label?) -> Lei's original work was based on edge removal classification based on actual edges that were present at month M but not M+1 as ground truth

Delay classification is performance based (Passenger experience): under these topological conditions will a train on a trajectory to what extent will the train be delayed?
 
Lei's: Edge Removal, infrastructural failure "Under these conditions, does the connection physically exist?"


In [84]:
# # Inspecting columns to keep/drop for graph features
# display(con.execute("""
#     SELECT column_name 
#     FROM (DESCRIBE services_valid)
# """).df())

### Create clean monthly services dataset (run once)

DOWNLOAD: downloads/data/filtered/final_aggregated/services_monthly.parquet

In [ ]:
if snellius is True:
	con.execute("""
		CREATE OR REPLACE TABLE services_monthly_agg AS
		SELECT
			-- grouping keys
			source,
			target,
			DATE_TRUNC('month', service_date) AS service_month,

			-- static features
			FIRST(distance) AS distance,

			-- departure/arrival stats aggregated for month
			SUM(departure_delay_min) AS total_departure_delay_minutes,
			ROUND(AVG(departure_delay_min), 2) AS avg_departure_delay_minutes,
			COUNT(*) FILTER (WHERE departure_delay_min > 0) AS count_delayed_departure_services,

			SUM(arrival_delay_min) AS total_delay_minutes,
			ROUND(AVG(arrival_delay_min), 2) AS avg_delay_minutes,
			COUNT(*) FILTER (WHERE arrival_delay_min > 0) AS count_delayed_services,

			COUNT(*) AS total_services,
			ROUND(COUNT(*) FILTER (WHERE departure_delay_min > 0) * 1.0 / NULLIF(COUNT(*),0),4) AS ratio_departure_delayed,
			ROUND(COUNT(*) FILTER (WHERE arrival_delay_min > 0) * 1.0 / NULLIF(COUNT(*),0),4) AS ratio_arrival_delayed,

			COUNT(*) FILTER (WHERE has_platform_change IS TRUE) AS count_platform_changes,
			histogram(train_type) AS train_type_counts

		FROM services_valid
		GROUP BY 1,2,3
	""")
	print("✅ Aggregated table 'services_monthly_agg' created.")

if cdata_new:
	aggregated_path = filtered_output_root_folder / "final_aggregated/services_monthly.parquet"
	con.execute(f"COPY services_monthly_agg TO '{aggregated_path}' (FORMAT PARQUET)")
	print("Done! Saved as ", aggregated_path)

In [ ]:
if check_rowc:
	count_row_diff("services_valid", "services_monthly_agg")

# Feature Engineering


## Network Topology (Graph) Features 

### Calculate graph features
(Brent's Disruptions adaptation & optimized)

Graph structure of Stations (nodes) remained consistent for all graphs in prior work (brent), as he utilized the stations of the whole dataset to create the daily graphs (this result in including stations that are not connected, so he only alternated the graph structure in edges i.e. the connectivity)

-> we only create a graph that is connected (hourly stations), as total_services is >= 1, which exclude stations that are do not have service in that hour from the graph (i.e. they do not exist as nodes) essentially alternating the graph structure (both nodes and edges connectivity). 

We also return None for hourly 

Feature we utilize now: source, target, weight (total_services_hour)

Could probably utilize lagged features (t-1) for features of t
total_delay_minutes	
avg_delay_minutes
count_delayed_services 
count_platform_changes

In [ ]:
# 2. Get all valid Date-Hour combinations first
print("Fetching time slots...")
time_slots = con.execute(f"""
	SELECT DISTINCT service_month
	FROM services_monthly_agg
	ORDER BY 1
""").fetchall()

print(f"✅ Found {len(time_slots)} monthly slots to process.")

# 3. Initialize Storage
# Key will be tuple: (datetime.date(2023, 1, 1), 8)
hourly_node_features = {}
hourly_edge_features = {}

# 4. Helper: Build Graph for One Hour
def get_monthly_graph(service_month_val):
	query = f"""
		SELECT 
			source, 
			target,
			distance
		FROM services_monthly_agg
		WHERE service_month = '{service_month_val}'
	"""
	df = con.execute(query).df()
	
	if df.empty:
		return None

	# Fast Bulk Loading
	G = nx.from_pandas_edgelist(
		df, 
		source='source', 
		target='target', 
		edge_attr=['distance'], 
		create_using=nx.DiGraph()
	)
	return G

# 5. Helper: Extract Graph Features
def extract_graph_features(G):
	# --- Node Features (Directed) ---
	n_feats = {}
	for node in G.nodes():
		deg = G.degree(node)
				
		# Avg Distance (Safe division)
		# # We look at outgoing edges for distance
		# total_dist = sum(d['distance'] for _, _, d in G.out_edges(node, data=True))
		# avg_dist = total_dist / G.out_degree(node) if G.out_degree(node) > 0 else 0

		# 1. Avg Distance of trains LEAVING here (Out-degree)
		out_edges = list(G.out_edges(node, data='distance'))
		avg_dist_out = sum(d for _, _, d in out_edges) / len(out_edges) if out_edges else 0

		# 2. Avg Distance of trains ARRIVING here (In-degree)
		in_edges = list(G.in_edges(node, data='distance'))
		avg_dist_in = sum(d for _, _, d in in_edges) / len(in_edges) if in_edges else 0
		
		n_feats[node] = {
			'degree': deg,
			'avg_distance_inbound': avg_dist_in,
			'avg_distance_outbound': avg_dist_out,
			'avg_distance': avg_dist_in + avg_dist_out  # Total avg distance (In + Out)
		}

	# --- Edge Features (Undirected for Topology Metrics) ---
	# We create the undirected view ONCE per graph, not per edge
	G_undir = G.to_undirected()
	
	e_feats = {}
	for u, v, data in G.edges(data=True):
		# Common Neighbors
		cn = list(nx.common_neighbors(G_undir, u, v))
		
		# Calculate complex metrics safely
		try:
			# Note: These return iterators, so we list() them and grab the score
			jaccard = list(nx.jaccard_coefficient(G_undir, [(u, v)]))[0][2]
			adamic = list(nx.adamic_adar_index(G_undir, [(u, v)]))[0][2]
			res_alloc = list(nx.resource_allocation_index(G_undir, [(u, v)]))[0][2]
		except:
			jaccard, adamic, res_alloc = 0, 0, 0

		e_feats[(u, v)] = {
			'common_neighbors': len(cn),
			'jaccard_coefficient': jaccard,
			'preferential_attachment': G.degree(u) * G.degree(v),
			'adamic_adar_index': adamic,
			'resource_allocation_index': res_alloc,
			'distance': data['distance']
		}
		
	return n_feats, e_feats

Fetching time slots...


NameError: name 'con' is not defined

### Create monthly graph features
DOWNLOAD: (folder) downloads/data/filtered/graph_features_output/

In [ ]:
import pandas as pd
import os

# Configuration
CHUNK_SIZE = 8760

# Buffers (Temporary Lists)
node_buffer = []
edge_buffer = []
batch_count = 0

print(f" Processing {len(time_slots)} months. Saving to '{GRAPH_OUTPUT_DIR}'...")

# --- MAIN LOOP ---
# tqdm gives you a real-time progress bar with ETA
for i, (date_val, hour_val) in enumerate(time_slots):
    
    # A. Build Graph
    G = get_monthly_graph(date_val, hour_val)
    if G is None: continue 

    # B. Extract Features
    n_feats, e_feats = extract_graph_features(G)
    
    # C. Flatten & Buffer (Convert Dict -> List of Rows)
    # Process Node Features
    for node, feats in n_feats.items():
        node_buffer.append({
            'date': date_val,
            'hour': hour_val,
            'station': node,
            **feats  # Unpacks {'degree': 10, ...}
        })

    # Process Edge Features
    for (u, v), feats in e_feats.items():
        edge_buffer.append({
            'date': date_val,
            'hour': hour_val,
            'source': u,
            'target': v,
            **feats # Unpacks {'jaccard': 0.5, ...}
        })
        
    # D. Save Checkpoint (Batch Flush)
    if len(node_buffer) > 0 and (i + 1) % CHUNK_SIZE == 0:
        # Save Nodes
        pd.DataFrame(node_buffer).to_parquet(
            f"{GRAPH_OUTPUT_DIR}/nodes_part_{batch_count}.parquet", index=False
        )
        # Save Edges
        pd.DataFrame(edge_buffer).to_parquet(
            f"{GRAPH_OUTPUT_DIR}/edges_part_{batch_count}.parquet", index=False
        )
        
        # Clear Memory
        node_buffer = []
        edge_buffer = []
        batch_count += 1

# E. Final Flush (Save whatever is left)
if node_buffer:
    pd.DataFrame(node_buffer).to_parquet(f"{GRAPH_OUTPUT_DIR}/nodes_part_{batch_count}.parquet", index=False)
    pd.DataFrame(edge_buffer).to_parquet(f"{GRAPH_OUTPUT_DIR}/edges_part_{batch_count}.parquet", index=False)

print("✅ Done! Data saved to parquet files.")

### Join graph features with services (run once)

We read this back in as aggregated_hourly_graph_features in duckdb

DOWNLOAD: downloads/data/filtered/final_aggregated/services_hourly_graph_features.parquet

We merge by the following:
Join NODES (graph_nodes) on 
hour = arrival_hour
date = service_date
station = source

EDGES (graph_edges) on
hour = arrival_hour
date = service_date
source = source
target = target
drop distance
drop weight

In [47]:
graph_nodes_path = str(GRAPH_OUTPUT_DIR / "nodes_part_*.parquet")
graph_edges_path = str(GRAPH_OUTPUT_DIR / "edges_part_*.parquet")

# Load into DuckDB Views
con.execute(f"CREATE OR REPLACE VIEW graph_nodes AS SELECT * FROM read_parquet('{graph_nodes_path}')")
con.execute(f"CREATE OR REPLACE VIEW graph_edges AS SELECT * FROM read_parquet('{graph_edges_path}')")

# Sanity Check: Verify we actually found files and loaded rows
print("✅Done loading graph features into duckdb!")

✅Done loading graph features into duckdb!


In [ ]:
# ...existing code...
con.execute("""
	CREATE OR REPLACE VIEW services_monthly_graph_features AS
	SELECT
		base.*,
		src.degree              AS src_degree,
		src.avg_distance        AS src_avg_distance,
		src.avg_distance_inbound AS src_avg_distance_in,
		src.avg_distance_outbound AS src_avg_distance_out,
		tgt.degree              AS tgt_degree,
		tgt.avg_distance        AS tgt_avg_distance,
		tgt.avg_distance_inbound AS tgt_avg_distance_in,
		tgt.avg_distance_outbound AS tgt_avg_distance_out,
		e.common_neighbors,
		e.jaccard_coefficient,
		e.preferential_attachment,
		e.adamic_adar_index,
		e.resource_allocation_index
	FROM aggregated_monthly base
	LEFT JOIN graph_nodes src
		ON base.service_month = src.month
		AND base.source = src.station
	LEFT JOIN graph_nodes tgt
		ON base.service_month = tgt.month
		AND base.target = tgt.station
	LEFT JOIN graph_edges e
		ON base.service_month = e.month
		AND base.source = e.source
		AND base.target = e.target
""")
print("✅ View 'services_monthly_graph_features' created successfully.")

# Save as parquet file 
if cdata_new:
	graph_features_output_path = filtered_output_root_folder / "final_aggregated/services_hourly_graph_features.parquet"
	con.execute(f"COPY services_hourly_graph_features TO '{graph_features_output_path}' (FORMAT PARQUET)")
	print("Done! Saved as ", graph_features_output_path)

✅ View 'services_hourly_graph_features' created successfully.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Done! Saved as  /home/jbao/NS_Thesis/downloads/data/filtered/final_aggregated/services_hourly_graph_features.parquet


# TODO CONTINUE HERE